# 01 Recopilación y transformación de datos (ETL)

Este notebook desarrolla la **Fase 1** del proyecto integrador:  
recopilación, limpieza y transformación del dataset de cambio climático y fenómeno ENSO  
publicado por el Gobierno de Colombia en [datos.gov.co](https://www.datos.gov.co/dataset/Cambio-clim-tico/bgt6-7zyr/about_data).

**Objetivo:** Obtener una base de datos confiable y enriquecida, lista para el análisis exploratorio (EDA), el dashboard de BI y el modelo predictivo.

---
**Dataset:** `Cambio_climatico.csv`  
**Fuente:** datos.gov.co — Indicador de Cambio Climático (IDEAM / Ministerio de Salud)  
**Cobertura:** Enero 1950 – Abril 2026 · Frecuencia mensual  
**Descripción:** Indicador que visualiza el comportamiento del fenómeno ENSO (El Niño/La Niña), las variaciones de temperatura superficial del océano Pacífico ecuatorial y su relación con eventos climáticos extremos en Colombia.


## 1. Preparación del entorno

In [30]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT           = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_PATH       = ROOT / 'data' / 'raw'       / 'Cambio_climatico.csv'
CLEANED_PATH   = ROOT / 'data' / 'cleaned'   / 'enso_cleaned.csv'
PROCESSED_PATH = ROOT / 'data' / 'processed' / 'enso_model_ready.csv'

print("Rutas configuradas:")
for name, path in [("RAW", RAW_PATH), ("CLEANED", CLEANED_PATH), ("PROCESSED", PROCESSED_PATH)]:
    print(f"  {name}: {path}")


Rutas configuradas:
  RAW: C:\Users\valex\PycharmProjects\Cambio_climatico_CO\data\raw\Cambio_climatico.csv
  CLEANED: C:\Users\valex\PycharmProjects\Cambio_climatico_CO\data\cleaned\enso_cleaned.csv
  PROCESSED: C:\Users\valex\PycharmProjects\Cambio_climatico_CO\data\processed\enso_model_ready.csv


## 2. Fuente de datos

El dataset proviene del portal de datos abiertos del Gobierno de Colombia (**datos.gov.co**), publicado por el IDEAM en el marco del indicador de *Cambio Climático*.  

Contiene registros **mensuales desde enero de 1950 hasta abril de 2026** e incluye:

| Columna original | Descripción |
|---|---|
| `date` | Fecha del registro (primer día de cada mes) |
| `TOTAL` | Temperatura Superficial del Mar (TSM) en la región Niño 3.4 del Pacífico (°C) |
| `ClimAdjust` | Línea base climatológica mensual — promedio histórico TSM para ese mes (°C) |
| `ANOM` | Anomalía mensual ENSO: `TOTAL - ClimAdjust` (°C) |
| `ANOM_trimester` | Índice ONI — promedio hacia adelante de 3 meses consecutivos de `ANOM` (°C) |
| `ANOM_trimester_round` | Índice ONI redondeado a 1 decimal |
| `phase_trimester` | Fase ENSO según ONI trimestral (`elnino` / `lanina` / `neutral`) |
| `phase_event` | Fase ENSO del evento activo en ese mes |

> **Nota metodológica:** El índice `ANOM_trimester` (ONI) se calcula como el promedio de los meses `t`, `t+1` y `t+2`, por lo que los 2 últimos registros del dataset (mar-abr 2026) no tienen ONI computable al no existir aún datos de mayo y junio 2026.


## 3. Extracción

In [31]:
raw_df = pd.read_csv(RAW_PATH)

display(raw_df.head(10))
print(f"Filas    : {raw_df.shape[0]:,}")
print(f"Columnas : {raw_df.shape[1]}")
print(f"Columnas : {list(raw_df.columns)}")


,date,TOTAL,ClimAdjust,ANOM,ANOM_trimester,ANOM_trimester_round,phase_trimester,phase_event
0,1950-01-01 00:00:00,24.56,26.18,-1.62,-1.336667,-1.3,lanina,lanina
1,1950-02-01 00:00:00,25.07,26.39,-1.32,-1.166667,-1.2,lanina,lanina
2,1950-03-01 00:00:00,25.88,26.95,-1.07,-1.183333,-1.2,lanina,lanina
3,1950-04-01 00:00:00,26.29,27.39,-1.11,-1.073333,-1.1,lanina,lanina
4,1950-05-01 00:00:00,26.19,27.56,-1.37,-0.850000,-0.8,lanina,lanina
5,1950-06-01 00:00:00,26.47,27.21,-0.74,-0.533333,-0.5,lanina,lanina
6,1950-07-01 00:00:00,26.28,26.72,-0.44,-0.423333,-0.4,neutral,neutral
7,1950-08-01 00:00:00,25.88,26.30,-0.42,-0.383333,-0.4,neutral,neutral
8,1950-09-01 00:00:00,25.73,26.14,-0.41,-0.443333,-0.4,neutral,neutral
9,1950-10-01 00:00:00,25.68,26.01,-0.32,-0.600000,-0.6,lanina,neutral


Filas    : 916
Columnas : 8
Columnas : ['date', 'TOTAL', 'ClimAdjust', 'ANOM', 'ANOM_trimester', 'ANOM_trimester_round', 'phase_trimester', 'phase_event']


> Se carga el archivo CSV original sin ninguna transformación. Se obtienen **916 registros** y **8 columnas**, cubriendo el periodo enero 1950 – abril 2026.


## 4. Calidad de los datos

In [32]:
print("Tipos de datos:")
print(raw_df.dtypes.astype(str))
print()
print("Valores nulos por columna:")
nulls = raw_df.isnull().sum()
pct   = (raw_df.isnull().mean() * 100).round(2)
null_report = pd.DataFrame({'Nulos': nulls, 'Porcentaje (%)': pct})
display(null_report[null_report['Nulos'] > 0])
print()
print("Revisión de filas con nulos:")
display(raw_df[raw_df.isnull().any(axis=1)])
print()
print(f"Duplicados exactos : {raw_df.duplicated().sum()}")
print(f"Rango de fechas    : {raw_df['date'].min()}  →  {raw_df['date'].max()}")
print(f"Valores únicos en phase_event    : {raw_df['phase_event'].unique()}")
print(f"Valores únicos en phase_trimester: {raw_df['phase_trimester'].unique()}")


Tipos de datos:
date                        str
TOTAL                   float64
ClimAdjust              float64
ANOM                    float64
ANOM_trimester          float64
ANOM_trimester_round    float64
phase_trimester             str
phase_event                 str
dtype: str

Valores nulos por columna:


,Nulos,Porcentaje (%)
ANOM_trimester,2,0.22
ANOM_trimester_round,2,0.22



Revisión de filas con nulos:


,date,TOTAL,ClimAdjust,ANOM,ANOM_trimester,ANOM_trimester_round,phase_trimester,phase_event
914,2026-03-01 00:00:00,27.28,27.19,0.09,NaN,NaN,neutral,neutral
915,2026-04-01 00:00:00,28.06,27.71,0.36,NaN,NaN,neutral,neutral



Duplicados exactos : 0
Rango de fechas    : 1950-01-01 00:00:00  →  2026-04-01 00:00:00
Valores únicos en phase_event    : <StringArray>
['lanina', 'neutral', 'elnino']
Length: 3, dtype: str
Valores únicos en phase_trimester: <StringArray>
['lanina', 'neutral', 'elnino']
Length: 3, dtype: str


> Se identifican **2 filas con nulos** (filas 914 y 915, correspondientes a **marzo y abril 2026**) en las columnas `ANOM_trimester` y `ANOM_trimester_round`.  
> Esto es **esperado y correcto**: el ONI requiere los meses `t+1` y `t+2` para calcularse, y esos datos no existen aún para los meses más recientes.  
> No se detectan duplicados. Las categorías de fase son consistentes: `elnino`, `lanina`, `neutral`.


## 5. Estadística descriptiva

In [33]:
display(raw_df.describe(include='all').T)

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
date,916,916,1950-01-01 00:00:00,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
TOTAL,916.0,NaN,NaN,NaN,26.903504,0.983302,24.25,26.21,26.98,27.59,29.41
ClimAdjust,916.0,NaN,NaN,NaN,26.877489,0.499353,26.01,26.5,26.73,27.28,27.94
ANOM,916.0,NaN,NaN,NaN,0.02619,0.852155,-2.09,-0.56,-0.035,0.53,2.77
ANOM_trimester,914.0,NaN,NaN,NaN,0.027615,0.829764,-2.033333,-0.545833,-0.045,0.545833,2.753333
ANOM_trimester_round,914.0,NaN,NaN,NaN,0.026477,0.829085,-2.0,-0.5,-0.0,0.5,2.8
phase_trimester,916,3,neutral,394,NaN,NaN,NaN,NaN,NaN,NaN,NaN
phase_event,916,3,neutral,421,NaN,NaN,NaN,NaN,NaN,NaN,NaN


- La anomalía ENSO mensual (`ANOM`) oscila entre **-2.09 °C** (La Niña más intensa, 1973) y **+2.77 °C** (El Niño más intenso, 2015).
- La temperatura base del Pacífico (`ClimAdjust`) varía entre 26.01 °C y 27.94 °C según el mes del año, reflejando la estacionalidad oceánica.
- La media de `ANOM` es ≈ 0.03 °C, lo que indica un leve sesgo positivo (calentamiento del Pacífico en el periodo 1950–2026).


## 6. Limpieza de datos

In [34]:
cleaned_df = raw_df.copy()

# ── 6.1 Conversión de fecha
cleaned_df['date'] = pd.to_datetime(cleaned_df['date'])

# ── 6.2 Renombrar columnas (inglés → español, descriptivo) ───
COLUMN_MAP = {
    'date'                 : 'Fecha',
    'TOTAL'                : 'Temperatura_Pacifico_C',
    'ClimAdjust'           : 'Temperatura_Base_C',
    'ANOM'                 : 'Anomalia_C',
    'ANOM_trimester'       : 'ONI_C',
    'ANOM_trimester_round' : 'ONI_Round',
    'phase_trimester'      : 'Fase_ONI',
    'phase_event'          : 'Fase_Evento',
}
cleaned_df.rename(columns=COLUMN_MAP, inplace=True)

# ── 6.3 Estandarización de categorías ────────────────────────
FASE_MAP = {
    'elnino'  : 'El Nino',
    'lanina'  : 'La Nina',
    'neutral' : 'Neutral',
}
cleaned_df['Fase_ONI']    = cleaned_df['Fase_ONI'].map(FASE_MAP)
cleaned_df['Fase_Evento'] = cleaned_df['Fase_Evento'].map(FASE_MAP)

print("Categorías estandarizadas:")
print(cleaned_df['Fase_Evento'].value_counts(dropna=False))
print()
print(f"Filas totales tras renombrar y estandarizar: {len(cleaned_df):,}")


Categorías estandarizadas:
Fase_Evento
Neutral    421
El Nino    249
La Nina    246
Name: count, dtype: int64

Filas totales tras renombrar y estandarizar: 916


- Se convierten las fechas a tipo datetime, se renombran las columnas al español con nombres descriptivos, y se estandarizan las categorías de fase ENSO (elnino - El Nino, lanina - La Nina, neutral - Neutral).
- **ClimAdjust se renombra a Temperatura_Base_C** para reflejar con precisión su significado: la temperatura climatológica de referencia mensual, no un ajuste arbitrario.
- **ANOM_trimester se renombra a ONI_C** (Oceanic Niño Index), que es el nombre estándar de este indicador a nivel internacional.


## 7. Tratamiento de nulos

In [35]:
print("Filas con nulos antes del tratamiento:")
display(cleaned_df[cleaned_df.isnull().any(axis=1)][['Fecha','Temperatura_Pacifico_C','Anomalia_C','ONI_C','Fase_ONI','Fase_Evento']])

rows_before = len(cleaned_df)

# Se eliminan las 2 filas de mar-abr 2026 porque su ONI no es computable
# (requieren datos de may-jun 2026, aún no disponibles).
# Todas las demás columnas de esas filas son válidas, pero sin ONI
# no es posible clasificar la fase ni calcular la intensidad del evento.
cleaned_df = cleaned_df.dropna(subset=['ONI_C']).reset_index(drop=True)

rows_after   = len(cleaned_df)
rows_removed = rows_before - rows_after

print(f"\nFilas antes  : {rows_before:,}")
print(f"Filas después: {rows_after:,}")
print(f"Eliminadas   : {rows_removed}  (marzo y abril 2026 — ONI incompleto)")
print(f"Nulos restantes: {cleaned_df.isnull().sum().sum()}")


Filas con nulos antes del tratamiento:


,Fecha,Temperatura_Pacifico_C,Anomalia_C,ONI_C,Fase_ONI,Fase_Evento
914,2026-03-01,27.28,0.09,NaN,Neutral,Neutral
915,2026-04-01,28.06,0.36,NaN,Neutral,Neutral



Filas antes  : 916
Filas después: 914
Eliminadas   : 2  (marzo y abril 2026 — ONI incompleto)
Nulos restantes: 0


- Se eliminan **2 filas** (marzo y abril 2026) cuyo ONI trimestral no puede calcularse porque los meses futuros necesarios aún no existen en el dataset.
- Al descartar solo la columna ONI_C como criterio (dropna(subset=['ONI_C'])), la eliminación es explícita y trazable.
- El dataset limpio queda con **914 registros** y **0 nulos**.


## 8. Variables derivadas

In [36]:
processed_df = cleaned_df.copy()

# ── 8.1 Variables temporales ─────────────────────────────────
processed_df['Anio']      = processed_df['Fecha'].dt.year
processed_df['Mes']       = processed_df['Fecha'].dt.month
processed_df['Trimestre'] = processed_df['Fecha'].dt.quarter
processed_df['Decada']    = (processed_df['Anio'] // 10 * 10).astype(str) + 's'

print("Variables temporales creadas: Anio, Mes, Trimestre, Decada")
print(processed_df[['Fecha','Anio','Mes','Trimestre','Decada']].head(5))


Variables temporales creadas: Anio, Mes, Trimestre, Decada
       Fecha  Anio  Mes  Trimestre Decada
0 1950-01-01  1950    1          1  1950s
1 1950-02-01  1950    2          1  1950s
2 1950-03-01  1950    3          1  1950s
3 1950-04-01  1950    4          2  1950s
4 1950-05-01  1950    5          2  1950s


- Se extraen componentes temporales de la fecha para facilitar análisis por año, mes, trimestre y década.
- Estas variables son esenciales para las visualizaciones de tendencias y el dashboard de BI.


In [37]:
# ── 8.2 Intensidad del evento (estándar NOAA, basado en ONI) ─
# La NOAA clasifica la intensidad usando el índice ONI (promedio trimestral),
# NO la anomalía mensual puntual. Se usa ONI_C para garantizar coherencia
# con el estándar internacional.

def clasificar_intensidad(oni, fase):
    """Clasifica la intensidad del evento ENSO según el ONI (estándar NOAA)."""
    if fase == 'Neutral':
        return 'Neutral'
    a = abs(oni)
    if a < 1.0:
        return 'Debil'
    elif a < 1.5:
        return 'Moderado'
    elif a < 2.0:
        return 'Fuerte'
    else:
        return 'Muy Fuerte'

processed_df['Intensidad_Evento'] = processed_df.apply(
    lambda row: clasificar_intensidad(row['ONI_C'], row['Fase_Evento']),
    axis=1
)

print("Distribución de intensidades:")
print(processed_df['Intensidad_Evento'].value_counts())


Distribución de intensidades:
Intensidad_Evento
Neutral       419
Debil         304
Moderado      116
Fuerte         57
Muy Fuerte     18
Name: count, dtype: int64


Se clasifica la intensidad de cada mes usando el **ONI trimestral** (`ONI_C`), conforme al estándar de la NOAA:
- `Débil` (|ONI| < 1.0) · `Moderado` (1.0–1.5) · `Fuerte` (1.5–2.0) · `Muy Fuerte` (≥ 2.0) · `Neutral`.
- **Corrección respecto a versiones anteriores:** se usa `ONI_C` en lugar de `Anomalia_C` mensual, eliminando las discrepancias de clasificación en ~108 registros.


In [38]:
# ── 8.3 Duración del evento ──────────────────────────────────
processed_df['_Cambio_Fase'] = (
    processed_df['Fase_Evento'] != processed_df['Fase_Evento'].shift(1)
).astype(int)

processed_df['_Grupo_Evento'] = processed_df['_Cambio_Fase'].cumsum()

duraciones = (
    processed_df
    .groupby('_Grupo_Evento')
    .size()
    .reset_index(name='Duracion_Meses')
)

processed_df = processed_df.merge(duraciones, on='_Grupo_Evento')
processed_df.drop(columns=['_Cambio_Fase', '_Grupo_Evento'], inplace=True)

print(f"Filas tras merge de duración: {len(processed_df)} (debe ser 914)")
print("Duraciones más frecuentes (meses):", sorted(processed_df['Duracion_Meses'].unique())[:10], "...")


Filas tras merge de duración: 914 (debe ser 914)
Duraciones más frecuentes (meses): [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)] ...


- Se calcula la **duración continua de cada evento ENSO** contando los meses consecutivos en la misma fase.
- El merge sobre el grupo de evento asigna el mismo valor de duración a todos los meses pertenecientes al mismo episodio.
- Se verifica que el número de filas se mantiene en **914** (sin duplicados generados por el merge).


In [39]:
# ── 8.4 Evento extremo ───────────────────────────────────────
# Se considera extremo cuando |ONI| >= 1.5 C (eventos Fuerte o Muy Fuerte)
processed_df['Evento_Extremo'] = (
    processed_df['ONI_C'].abs() >= 1.5
).astype(int)

print(f"Meses con evento extremo (|ONI| >= 1.5°C): {processed_df['Evento_Extremo'].sum()}")
print(f"Porcentaje del total: {processed_df['Evento_Extremo'].mean()*100:.1f}%")


Meses con evento extremo (|ONI| >= 1.5°C): 75
Porcentaje del total: 8.2%


- Se crea un indicador binario `Evento_Extremo` (1 = extremo, 0 = no extremo) usando el umbral |ONI| ≥ 1.5 °C, que corresponde a eventos **Fuertes o Muy Fuertes** según la NOAA.
- Estos son los episodios de mayor impacto histórico en Colombia.


In [40]:
# ── 8.5 Tendencia móvil 12 meses ────────────────────────────
# Promedio móvil anual de la anomalía mensual.
# Los primeros 5 registros quedan en NaN porque se requieren
# mínimo 6 meses para calcular el promedio (min_periods=6).
# Esto es esperado y se documenta explícitamente.
processed_df['Anomalia_12m'] = (
    processed_df['Anomalia_C']
    .rolling(window=12, min_periods=6)
    .mean()
    .round(3)
)

# ── 8.6 Variación mensual ────────────────────────────────────
# Cambio de la anomalía respecto al mes anterior.
# La primera fila queda en NaN por definición (no hay mes previo).
processed_df['Anomalia_Delta'] = (
    processed_df['Anomalia_C']
    .diff()
    .round(3)
)

nulos_derivadas = processed_df[['Anomalia_12m','Anomalia_Delta']].isnull().sum()
print("Nulos en variables de tendencia (esperados por cálculo matemático):")
print(nulos_derivadas)
print()
print("Detalle Anomalia_12m — primeras filas:")
print(processed_df[['Fecha','Anomalia_C','Anomalia_12m','Anomalia_Delta']].head(8).to_string())


Nulos en variables de tendencia (esperados por cálculo matemático):
Anomalia_12m      5
Anomalia_Delta    1
dtype: int64

Detalle Anomalia_12m — primeras filas:
       Fecha  Anomalia_C  Anomalia_12m  Anomalia_Delta
0 1950-01-01       -1.62           NaN             NaN
1 1950-02-01       -1.32           NaN            0.30
2 1950-03-01       -1.07           NaN            0.25
3 1950-04-01       -1.11           NaN           -0.04
4 1950-05-01       -1.37           NaN           -0.26
5 1950-06-01       -0.74        -1.205            0.63
6 1950-07-01       -0.44        -1.096            0.30
7 1950-08-01       -0.42        -1.011            0.02


- **`Anomalia_12m`**: promedio móvil de 12 meses con mínimo 6 observaciones. Los primeros **5 registros** (ene–may 1950) resultan en `NaN` porque no acumulan los 6 meses requeridos — comportamiento matemático normal del `rolling`.
- **`Anomalia_Delta`**: diferencia mes a mes. El **primer registro** (ene 1950) resulta en `NaN` porque no existe un mes anterior — comportamiento estándar de `diff()`.
- En total quedan **6 nulos** en el dataset procesado, todos ellos **justificados y esperados** por la naturaleza de estos cálculos. No se imputan ni eliminan.


## 9. Reordenamiento de columnas

In [41]:
FINAL_COLUMNS = [
    # Temporales
    'Fecha', 'Anio', 'Mes', 'Trimestre', 'Decada',
    # Temperatura
    'Temperatura_Pacifico_C', 'Temperatura_Base_C',
    # Anomalía e índice ONI
    'Anomalia_C', 'ONI_C', 'ONI_Round',
    # Tendencias
    'Anomalia_12m', 'Anomalia_Delta',
    # Clasificación ENSO
    'Fase_ONI', 'Fase_Evento', 'Intensidad_Evento',
    # Métricas de evento
    'Duracion_Meses', 'Evento_Extremo',
]

processed_df = processed_df[FINAL_COLUMNS]

print("Columnas finales:", list(processed_df.columns))
print(f"Shape final: {processed_df.shape}")


Columnas finales: ['Fecha', 'Anio', 'Mes', 'Trimestre', 'Decada', 'Temperatura_Pacifico_C', 'Temperatura_Base_C', 'Anomalia_C', 'ONI_C', 'ONI_Round', 'Anomalia_12m', 'Anomalia_Delta', 'Fase_ONI', 'Fase_Evento', 'Intensidad_Evento', 'Duracion_Meses', 'Evento_Extremo']
Shape final: (914, 17)


- Se reordenan las columnas en grupos lógicos: temporales - temperatura - anomalías e índice ONI -tendencias - clasificación ENSO - métricas de evento.
- Este orden facilita la lectura del dataset y su uso en herramientas de BI.


## 10. Validación final

In [42]:
print("=" * 55)
print("VALIDACIÓN DEL DATASET PROCESADO")
print("=" * 55)
print(f"Filas           : {processed_df.shape[0]:,}  (raw: 916 → -2 filas ONI incompleto)")
print(f"Columnas        : {processed_df.shape[1]}")
print(f"Periodo         : {processed_df['Fecha'].min().strftime('%Y-%m')}  →  {processed_df['Fecha'].max().strftime('%Y-%m')}")
print(f"Duplicados      : {processed_df.duplicated().sum()}")
print()
print("Nulos por columna:")
nulls_final = processed_df.isnull().sum()
display(pd.DataFrame({
    'Columna': nulls_final.index,
    'Nulos': nulls_final.values,
    'Origen': [
        '—','—','—','—','—',
        '—','—',
        '—','—','—',
        'rolling(12, min_periods=6) — primeros 5 meses sin ventana completa',
        'diff() — primer mes sin mes anterior',
        '—','—','—',
        '—','—'
    ]
}).query("Nulos > 0"))
print()
print("Distribución de fases (Fase_Evento):")
print(processed_df['Fase_Evento'].value_counts())
print()
print("Distribución de intensidades:")
print(processed_df['Intensidad_Evento'].value_counts())
print()
print("Vista previa del dataset final:")
display(processed_df.head(5))


VALIDACIÓN DEL DATASET PROCESADO
Filas           : 914  (raw: 916 → -2 filas ONI incompleto)
Columnas        : 17
Periodo         : 1950-01  →  2026-02
Duplicados      : 0

Nulos por columna:


,Columna,Nulos,Origen
10,Anomalia_12m,5,"rolling(12, min_periods=6) — primeros 5 meses ..."
11,Anomalia_Delta,1,diff() — primer mes sin mes anterior



Distribución de fases (Fase_Evento):
Fase_Evento
Neutral    419
El Nino    249
La Nina    246
Name: count, dtype: int64

Distribución de intensidades:
Intensidad_Evento
Neutral       419
Debil         304
Moderado      116
Fuerte         57
Muy Fuerte     18
Name: count, dtype: int64

Vista previa del dataset final:


,Fecha,Anio,Mes,Trimestre,Decada,Temperatura_Pacifico_C,Temperatura_Base_C,Anomalia_C,ONI_C,ONI_Round,Anomalia_12m,Anomalia_Delta,Fase_ONI,Fase_Evento,Intensidad_Evento,Duracion_Meses,Evento_Extremo
0,1950-01-01,1950,1,1,1950s,24.56,26.18,-1.62,-1.336667,-1.3,NaN,NaN,La Nina,La Nina,Moderado,6,0
1,1950-02-01,1950,2,1,1950s,25.07,26.39,-1.32,-1.166667,-1.2,NaN,0.30,La Nina,La Nina,Moderado,6,0
2,1950-03-01,1950,3,1,1950s,25.88,26.95,-1.07,-1.183333,-1.2,NaN,0.25,La Nina,La Nina,Moderado,6,0
3,1950-04-01,1950,4,2,1950s,26.29,27.39,-1.11,-1.073333,-1.1,NaN,-0.04,La Nina,La Nina,Moderado,6,0
4,1950-05-01,1950,5,2,1950s,26.19,27.56,-1.37,-0.850000,-0.8,NaN,-0.26,La Nina,La Nina,Debil,6,0


- El dataset final tiene **914 filas × 17 columnas**, sin duplicados y con únicamente **6 nulos justificados** en variables derivadas de cálculo.
- La distribución de fases y de intensidades es coherente con el histórico ENSO documentado por la NOAA para el Pacífico ecuatorial.


## 11. Carga de resultados

In [43]:
CLEANED_PATH.parent.mkdir(parents=True, exist_ok=True)
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)

cleaned_df.to_csv(CLEANED_PATH, index=False)
processed_df.to_csv(PROCESSED_PATH, index=False)

print(f"✓ Dataset limpio guardado   → {CLEANED_PATH.name}  ({len(cleaned_df):,} filas)")
print(f"✓ Dataset procesado guardado → {PROCESSED_PATH.name}  ({len(processed_df):,} filas)")


✓ Dataset limpio guardado   → enso_cleaned.csv  (914 filas)
✓ Dataset procesado guardado → enso_model_ready.csv  (914 filas)


## 12. Resumen de la Fase 1

**Decisiones metodológicas clave:**
- La intensidad del evento se clasifica con el **ONI trimestral** (no la anomalía mensual puntual), conforme al estándar internacional NOAA.
- Los 6 nulos residuales en variables de tendencia **no se imputan** porque corresponden al inicio de la serie donde no existe información histórica previa suficiente.
- Se mantiene la cobertura 1950–2026 (76 años) para capturar el máximo contexto histórico del ENSO y su relación con Colombia.
